# Study report: irq-scenarios

The study measures the four scenarios of the upstream interrupt benchmark
(Zephyr PR 117170) that the study did not have: `irq_unlock()` to the ISR
of a pending interrupt, the nested entry, the exit to the interrupted
thread and back-to-back ISRs. The entry from a busy thread is the
reference. The app page of `irq_latency` (section Scenarios) defines
each span.

This notebook reads one built instance of the study (`study.py analysis
build`). `study.py analysis notebooks` runs it and writes the HTML into the
instance folder. In the figures: hover for values, drag to zoom, click a
legend entry to hide a series, and use the menu above a figure to select
the metric. Commit this file without outputs.

In [ ]:
# Parameters. The instance folder: LATBENCH_INSTANCE ("study.py analysis notebooks" sets it), else the
# folder of this notebook when it is the executed copy in <instance>/notebooks/, else the instance of this
# study in LATBENCH_RESULTS (a results root). The safety repo: LATBENCH_REPO, else the design repo of the
# lab space of the instance (.lab/config), else the repo of the current folder.
import configparser
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
INSTANCE = Path(os.environ.get("LATBENCH_INSTANCE") or next(
    (d for d in (here, here.parent) if (d / "study.json").is_file() and (d / "instance.json").is_file()),
    Path(os.environ.get("LATBENCH_RESULTS", "/wrk/z/lab-kernel-latency/results/kernel-latency")) / "st-default-latencies@nucleo_g474re/st-irq-scenarios"))
lab = next((d for d in INSTANCE.resolve().parents if (d / ".lab" / "config").is_file()), None)
if os.environ.get("LATBENCH_REPO"):
    REPO = Path(os.environ["LATBENCH_REPO"])
elif lab:
    cfg = configparser.ConfigParser()
    cfg.read(lab / ".lab" / "config")
    REPO = lab / cfg.get("design", "repo", fallback="safety")
else:
    REPO = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True,
                               check=True).stdout.strip())

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "scripts"))
from probatrix.benchmarks import data  # noqa: E402
from probatrix.benchmarks import nbfig  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
nbfig.intro(inst)

## Central results

In [ ]:
inst.central()

## Hypotheses

A hypothesis is one claim about a mechanism. It is wrong if one of its
checks is wrong. The statements and the limits were written before the
runs; the column `basis` says where each number comes from.

In [ ]:
inst.hypotheses()

In [ ]:
inst.checks()

## All scenarios (irq-scenarios-1m)

Upstream numbers on the same board (PR runs, p50, regular path, flash):
`locked_unlock_to_isr` 87..112, `nested_preempt` 77,
`exit_resume_interrupted` 56..69, `throughput_round_trip` 102..142,
`entry_trigger_to_isr` 107..128, `entry_direct_isr` 50..72.

In [ ]:
if nbfig.built(inst, "irq-scenarios-1m"):
    df = inst.rows("irq-scenarios-1m")
    nbfig.dots(df, "IRQ scenarios (irq-scenarios-1m)", stats=("p50", "p99", "p99.99", "max")).show()
    display(nbfig.pivot(df, ["min", "p50", "p99", "p99.99", "max"]))

In [ ]:
if nbfig.built(inst, "irq-scenarios-1m"):
    series = nbfig.hists_of(inst, "irq-scenarios-1m")
    # The points have different metrics: one figure per metric, with the points that have it.
    for m in ["irq", "irq_sw", "irq_unlock", "irq_nested", "irq_nested_sw", "irq_exit", "irq_b2b"]:
        sub = {k: v for k, v in series.items() if m in v}
        if sub:
            nbfig.ccdf(sub, f"{m}: CCDF (irq-scenarios-1m)", [m]).show()

## H2: the tail of irq_unlock and of the other scenarios

The witness rows of each scenario. A row with the cause `systick` had a
SysTick reload during the sample and extra exception cycles.

In [ ]:
if nbfig.built(inst, "irq-scenarios-1m"):
    w = inst.tails("irq-scenarios-1m")
    display(inst.tail_causes("irq-scenarios-1m"))
    if len(w):
        nbfig.tails(w, "Witnesses of the largest scenario samples (irq-scenarios-1m)").show()
        display(w.head(40))